# MCP Controlled Agentic Workflow

This notebook demonstrates a real MCP client/server interaction and a constrained router that selects one approved workflow based on natural language.

## Setup and imports

We import the demo package, verify the local server module path, and prepare the notebook to run in the repository root.

In [1]:
import os
import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
for _ in range(5):
    if (workspace_root / "src").exists():
        break
    workspace_root = workspace_root.parent
sys.path.insert(0, str(workspace_root / "src"))

from genai_demos.mcp_poc.mcp_helpers import with_mcp_session, call_tool, print_json, SERVER_PATH
from genai_demos.mcp_poc.router import route_with_llm
from genai_demos.mcp_poc.workflows import WORKFLOW_REGISTRY


## Server path sanity check

Confirm that the demo server module exists and can be referenced from this notebook.

In [2]:
print("MCP server path:", SERVER_PATH)
print("Server file exists:", SERVER_PATH.exists())


MCP server path: /Users/douglasdaly/GitHub/Generative-AI/src/genai_demos/mcp_poc/server.py
Server file exists: True


## OpenAI key and fallback behavior

The router uses the OpenAI API when a key is available. If `OPENAI_API_KEY` is missing, the notebook falls back to a deterministic router.

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

print("Using OPENAI_API_KEY:", bool(os.environ.get("OPENAI_API_KEY")))


Using OPENAI_API_KEY: True


## Connect to the MCP server

We launch a local MCP stdio session and verify that the client can connect to the server.

In [4]:
async def probe_server():
    async def callback(session):
        return {"ok": True, "message": "MCP session established"}

    return await with_mcp_session(callback)

result = await probe_server()
print_json(result)


{
  "ok": true,
  "message": "MCP session established"
}


## List demo resources and tools

These are the capabilities the demo server exposes. In a full MCP deployment, these would be discovered dynamically from the server.

In [5]:
async def inspect_capabilities():
    def normalize(value):
        if hasattr(value, "to_dict"):
            return normalize(value.to_dict())
        if isinstance(value, dict):
            return {k: normalize(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [normalize(v) for v in value]
        if hasattr(value, "__dict__"):
            return {k: normalize(v) for k, v in vars(value).items()}
        try:
            json.dumps(value)
            return value
        except (TypeError, OverflowError):
            return str(value)

    async def callback(session):
        resources = await session.list_resources()
        tools = await session.list_tools()
        return {
            "resources": normalize(resources),
            "tools": normalize(tools),
        }

    return await with_mcp_session(callback)

capabilities = await inspect_capabilities()
print_json(capabilities)


{
  "resources": {
    "meta": null,
    "nextCursor": null,
    "resources": [
      {
        "name": "customer_all",
        "title": null,
        "uri": {
          "_url": "customer://all"
        },
        "description": "",
        "mimeType": "text/plain",
        "size": null,
        "icons": null,
        "annotations": null,
        "meta": null
      },
      {
        "name": "inventory_categories",
        "title": null,
        "uri": {
          "_url": "inventory://categories"
        },
        "description": "",
        "mimeType": "text/plain",
        "size": null,
        "icons": null,
        "annotations": null,
        "meta": null
      }
    ]
  },
  "tools": {
    "meta": null,
    "nextCursor": null,
    "tools": [
      {
        "name": "lookup_customer",
        "title": null,
        "description": "",
        "inputSchema": {
          "properties": {
            "customer_id": {
              "title": "Customer Id",
              "type": "string"


## Call one tool directly

We use the live MCP session to invoke a single tool and inspect the result.

In [6]:
async def call_one_tool():
    async def callback(session):
        return await call_tool(session, "lookup_customer", customer_id="123")

    return await with_mcp_session(callback)

customer_result = await call_one_tool()
print_json(customer_result)


{
  "ok": true,
  "customer": {
    "customer_id": "123",
    "name": "Alex Rivera",
    "email": "alex.rivera@example.com",
    "segment": "student gamer",
    "preferences": [
      "portable",
      "gaming",
      "under $1500",
      "good battery life"
    ],
    "recent_purchases": [
      "USB-C dock",
      "wireless mouse"
    ]
  }
}
{
  "ok": true,
  "customer": {
    "customer_id": "123",
    "name": "Alex Rivera",
    "email": "alex.rivera@example.com",
    "segment": "student gamer",
    "preferences": [
      "portable",
      "gaming",
      "under $1500",
      "good battery life"
    ],
    "recent_purchases": [
      "USB-C dock",
      "wireless mouse"
    ]
  }
}


## Approved workflows

The router is constrained to a small set of approved workflows. This keeps the system predictable and safe.

In [7]:
print_json({
    "approved_workflows": list(WORKFLOW_REGISTRY.keys()),
})


{
  "approved_workflows": [
    "product_recommendation",
    "support_issue",
    "inventory_notification"
  ]
}


## Route a natural-language request

The router selects one approved workflow and fills required parameters. If no OpenAI key is available, the demo uses a deterministic fallback.

In [8]:
user_prompt = "Customer 123 has a laptop battery overheating issue. Create a support ticket and follow up."
route = route_with_llm(user_prompt)
print_json(route)


[06/22/26 19:56:01] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=902102;file:///opt/anaconda3/envs/ai_py312/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=321401;file:///opt/anaconda3/envs/ai_py312/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

{
  "workflow": "support_issue",
  "parameters": {
    "customer_id": "123",
    "issue": "laptop battery overheating"
  }
}


## Execute the selected workflow

The notebook executes the approved workflow selected by the router over the live MCP session.

In [9]:
async def run_workflow_and_inspect_activity():
    workflow_name = route["workflow"]
    workflow = WORKFLOW_REGISTRY[workflow_name]
    parameters = route["parameters"]

    async def callback(session):
        workflow_result = await workflow(session, **parameters)
        activity_response = await call_tool(session, "get_activity_log")
        return {
            "workflow_result": workflow_result,
            "activity_log": activity_response,
        }

    return await with_mcp_session(callback)

result = await run_workflow_and_inspect_activity()
print_json(result)


STEP 1: Look up customer
{
  "ok": true,
  "customer": {
    "customer_id": "123",
    "name": "Alex Rivera",
    "email": "alex.rivera@example.com",
    "segment": "student gamer",
    "preferences": [
      "portable",
      "gaming",
      "under $1500",
      "good battery life"
    ],
    "recent_purchases": [
      "USB-C dock",
      "wireless mouse"
    ]
  }
}

STEP 2: Create support ticket
{
  "ok": true,
  "ticket": {
    "ticket_id": "TICKET-0001",
    "customer_id": "123",
    "customer_email": "alex.rivera@example.com",
    "issue": "laptop battery overheating",
    "priority": "normal",
    "created_at": "2026-06-23T02:56:01.922972+00:00",
    "stub": true
  }
}

STEP 3: Generate follow-up email
{
  "ok": true,
  "email": {
    "to": "alex.rivera@example.com",
    "subject": "Support Ticket TICKET-0001",
    "body": "Hello Alex Rivera,\n\nYour support ticket has been created.\n\nIssue: laptop battery overheating\nTicket ID: TICKET-0001\n\nThank you.",
    "sent_at": "202

## Inspect activity log

The workflow and the activity-log lookup must happen in the same MCP server session. The previous cell already does that, so this cell pulls the log out of the saved `result` object and displays it directly.

If this cell prints an empty log, rerun the workflow cell above first.


In [10]:
if "result" not in globals():
    print("No workflow result found. Run the previous workflow cell first.")
else:
    activity_log = result.get("activity_log", {})
    tickets = activity_log.get("tickets", [])
    emails = activity_log.get("emails", [])

    print(f"Tickets created in this session: {len(tickets)}")
    print(f"Emails simulated in this session: {len(emails)}")
    print_json(activity_log)


Tickets created in this session: 1
Emails simulated in this session: 1
{
  "tickets": [
    {
      "ticket_id": "TICKET-0001",
      "customer_id": "123",
      "customer_email": "alex.rivera@example.com",
      "issue": "laptop battery overheating",
      "priority": "normal",
      "created_at": "2026-06-23T02:56:01.922972+00:00",
      "stub": true
    }
  ],
  "emails": [
    {
      "to": "alex.rivera@example.com",
      "subject": "Support Ticket TICKET-0001",
      "body": "Hello Alex Rivera,\n\nYour support ticket has been created.\n\nIssue: laptop battery overheating\nTicket ID: TICKET-0001\n\nThank you.",
      "sent_at": "2026-06-23T02:56:01.924501+00:00",
      "stub": true
    }
  ]
}


## Takeaways

- MCP makes tool discovery explicit, not implicit.
- A constrained router keeps the workflow safe while still allowing natural-language intent.
- Live MCP discovery ensures the client and server agree on available resources and tools.
- The next step is to extend this pattern with richer workflows and production-safe tool metadata.